# PGLite: PostgreSQL in the Browser 🐘

This notebook demonstrates using [PGLite](https://github.com/electric-sql/pglite), a WebAssembly build of PostgreSQL, to run a full-featured Postgres database entirely in your browser.

First, we need to install the `pglite` package.

In [ ]:
# Install pglite using micropip
import micropip
await micropip.install('pglite')
print("pglite installed successfully.")

## 1. Connect to a Database

We can connect to an in-memory database or persist the data to the browser's IndexedDB for storage across sessions.

In [ ]:
from pglite import PGLite

# Use 'memory' for a temporary database, or a path for a persistent one.
# The path will be stored in the browser's IndexedDB.
db = PGLite(datadir="/tmp/pgdata")
print(f'PGLite connected. Postgres version: {db.version}')

## 2. Execute SQL Commands
Now we can run standard PostgreSQL commands.

In [ ]:
await db.query("DROP TABLE IF EXISTS employees;")
await db.query("CREATE TABLE employees (id serial PRIMARY KEY, name VARCHAR(50), role VARCHAR(50), salary NUMERIC(10, 2));")

print("Table 'employees' created.")

## 3. Insert and Query Data
Let's add some employees and query them.

In [ ]:
await db.query("INSERT INTO employees (name, role, salary) VALUES ('Alice', 'Engineer', 80000.00), ('Bob', 'Designer', 75000.00), ('Charlie', 'Manager', 95000.00);")

# Run a query and get the results
results = await db.query("SELECT * FROM employees ORDER BY salary DESC;")

print("Query Results:")
for row in results:
    print(row)

## 4. Use with Pandas
PGLite integrates with Pandas, but requires a different connection method using the `psycopg` driver.

In [ ]:
# PGLite provides a psycopg-compatible connection interface
import pandas as pd

conn = db.psyco_connect()
df = pd.read_sql('SELECT name, role FROM employees WHERE salary > 80000', conn)
conn.close()

print("High-Earning Employees:")
display(df)

Closing the main connection will save the data to IndexedDB if a datadir was specified.

In [ ]:
await db.close()
print("Database connection closed.")